In [1]:
from gmplib import *

In [177]:
def LazyPE(x):
    N = len(x.parent().tensor_factors())
    LSF = LazySymmetricFunctions(tensor([p]*N))
    return exp(LSF(lambda n: p[n](x)/n, valuation=1))

# Intertwiners

In [12]:
def philam(lam,x,y):
    x = coercion_safe(x,s)
    y = coercion_safe(y,s)
    return (y(s[1]-epsilon(lam,-1)/z/q3)).scalar_qt(x(s[1]+z*epsilon(lam)))

In [13]:
def testIntertwiner(mu,lam,nu):
    lhs = x2d(mu) * philam(lam,McdP(mu),McdP(nu))
    rhs1 = -1/z*sum(s[k-1]((1-1/q3)/z*x2d(lam,-1)) * philam(lam,McdP(mu),xplus_k(-k,McdP(nu))) for k in range(1,sum(nu)+1))
    rhs2 = sum(PE((chi2d(rho,-1)-chi2d(lam,-1))*x2d(lam)-1) * philam(rho,McdP(mu),McdP(nu)) for rho in Partition(lam).up())
    return lhs==(rhs1+rhs2)

In [4]:
all(testIntertwiner(mu,lam,nu) for nu in partitions_up_to_k(3) for lam in partitions_up_to_k(3) for mu in partitions_up_to_k(3))

True

In [148]:
def testLemma64(lam,n):
    lhs = 0
    rhs = + sum((chi2d(nu)-chi2d(lam))**(-n)*PE((chi2d(nu,-1)-chi2d(lam,-1))*x2d(lam)-1) for nu in Partition(lam).up())
    if n > 0:
        rhs += -s[n-1](x2d(lam,-1))
    else:
        lhs += s[-n](x2d(lam))
    return lhs == rhs

In [150]:
all(all(testLemma64(lam,n) for n in range(-5,5+1)) for lam in partitions_up_to_k(7))

True

In [136]:
def testLemma65(lam,n):
    if lam == []:
        return (1-q3/z) == -q3/z*(1-z/q3) + 0
    lhs = 0
    rhs = sum((chi2d(lam)-chi2d(mu))**(-n)*PE(-q3*(chi2d(lam,-1)-chi2d(mu,-1))*x2d(lam)-1) for mu in Partition(lam).down())
    if n >= -1:
        rhs += -q3*s[n+1](-x2d(lam,-1)/q3)
    if n <= 0:
        lhs += s[-n](-q3*x2d(lam))
    return lhs == rhs

In [138]:
all(all(testLemma65(lam,n) for n in range(-5,5+1)) for lam in partitions_up_to_k(5))

True

In [139]:
def theta_ast(lam,x,y,u):
    x = coercion_safe(x,s)
    y = coercion_safe(y,s)
    return (y(s[1]+epsilon(lam,-1)/u)).scalar_qt(x(s[1]-u*epsilon(lam)))

In [140]:
def testIntertwinerast(mu,lam,nu):
    x = coercion_safe(McdP(mu),s)
    y = coercion_safe(McdP(nu),s)
    lhs = x2d(mu) * theta_ast(lam,x,y,q3*z)
    rhs1 = theta_ast(lam,x,-q3*z*xplus_k(1,y),q3*z)
    rhs2 = sum(PE(-q3*(chi2d(lam,-1)-chi2d(rho,-1))*x2d(lam)-1) * theta_ast(rho,x(s[1]-q2*(1-q1)*(1-q3)*z*(chi2d(lam)-chi2d(rho))),y,q3*z) for rho in Partition(lam).down())
    return lhs == (rhs1+rhs2)

In [143]:
all(all(all(testIntertwinerast(mu,lam,nu) for mu in partitions_up_to_k(6)) for lam in partitions_up_to_k(6)) for nu in partitions_up_to_k(6))

True

In [157]:
all(all(rtildeast(lam,chi2d(nu)-chi2d(lam))*b_tilde(lam)==(-q)*b_tilde(nu)*rtilde(nu,chi2d(nu)-chi2d(lam)) for nu in lam.up()) for lam in partitions_up_to_k(6))

True

## $N=1$

In [73]:
# def testWhittakerEta(k,lam,d):
#     if d-k < 0:
#         lhs = s.zero()
#     else:
#         lhs = xplus_k(k,s[d-k]((1-t)/(1-q)*epsilon(lam)*s[1]))
#     rhs = sum((chi2d(nu)-chi2d(lam))**-k*PE((chi2d(nu,-1)-chi2d(lam,-1))*x2d(lam)-1)*s[d]((1-t)/(1-q)*epsilon(nu)*s[1]) for nu in Partition(lam).up())
#     rhs += -sum( s([k-j-1])(x2d(lam,-1))*s([j])((1-q2)*s[1])*s[d-j]((1-t)/(1-q)*epsilon(lam)*s[1]) for j in range(min(d,k-1)+1))
#     return lhs==rhs

In [74]:
# all(testWhittakerEta(k,lam,d) for lam in partitions_up_to_k(8) for k in range(8) for d in range(8))

In [75]:
# def Whittaker(lam,n):
#     return u[0]**sum(lam[0]) * s[n]((1-t)/(1-q)*u[0]*epsilon(lam[0])*tensor([p[1]]))

## $N=2$

In [13]:
# def MMM(lam,mu,nu):
#     tmp = epsilon(nu)*epsilon(mu,-1)-u[0]/u[1]/q3*epsilon(lam)*epsilon(mu,-1)+u[0]/u[1]/q3*epsilon(lam)*epsilon(nu,-1)
#     res = u[0]**(sum(lam)+sum(mu)-sum(nu))*(-q*u[1])**sum(nu)/DET(chi2d(nu))
#     res *= b_tilde(nu)
#     res *= PE((1-t)/(1-q)*tmp-1/(1-q1)/(1-q2))
#     return res

In [24]:
def MMM2(lam,mu,nu):
    res = u[0]**(sum(lam)+sum(mu)-sum(nu))*(-q*u[1])**sum(nu)/DET(chi2d(nu))
    res *= b_tilde(nu)
    res *= Nekrasov(nu,mu,1)*Nekrasov(lam,nu,u[0]/u[1]/q3)/Nekrasov(lam,mu,u[0]/u[1]/q3)
    return res

In [465]:
def MMM2(lam,mu,nu):
    res =  u[0]**(sum(lam)+sum(mu)-sum(nu)) * u[1]**sum(nu)
    res *= tNek(nu,mu,q3)
    res *= tNek(lam,nu,u[0]/u[1])
    res /= tNek(nu,nu,1)
    res /= tNek(lam,mu,u[0]/u[1])
    return res

In [466]:
def MMM2(lam,mu,nu):
    res = u[0]**(sum(lam)+sum(mu)) * (u[1]/u[0]/q3)**sum(nu)
    res *= NekrasovJEB(nu,mu,q3)
    res *= NekrasovJEB(lam,nu,u[0]/u[1])
    res /= NekrasovJEB(nu,nu,1)
    res /= NekrasovJEB(lam,mu,u[0]/u[1])
    return res

In [467]:
def MMM2(lam,mu,nu):
    res = u[0]**sum(lam) * (q3*u[1])**sum(mu) * q3**-sum(nu)
    res *= NekrasovJEB(mu,nu,1)
    res *= NekrasovJEB(nu,lam,q3*u[1]/u[0])
    res /= NekrasovJEB(nu,nu,1)
    res /= NekrasovJEB(mu,lam,q3*u[1]/u[0])
    return res

In [5]:
def MMM2(lam,mu,nu):
    res = u[0]**sum(lam) * (q3*u[1])**sum(mu)
    res *= NekrasovJEB(mu,nu,1)
    res *= NekrasovJEB(nu,lam,q3*u[1]/u[0])
    res /= (-1)**sum(nu) * tNek(nu,nu,1) * DET(chi2d(nu))
    res /= NekrasovJEB(mu,lam,q3*u[1]/u[0])
    return res

In [6]:
def Whittaker2(lam,n):
    if n == 0:
        return sum(MMM2(lam[0],lam[1],nu) for nu in partitions_up_to_k(sum(lam[0])+sum(lam[1]))) * tensor([p[0],p[0]])
    X1 = tensor([p[1],p[0]])
    X2 = tensor([p[0],p[1]])
    res = sum(MMM2(lam[0],lam[1],nu) * s[n]((1-t)/(1-q)*((u[0]*epsilon(lam[0])+u[1]*epsilon(lam[1])-q3*u[1]*epsilon(nu))*X1+u[1]*epsilon(nu)*X2)) for nu in partitions_up_to_k(sum(lam[0])+sum(lam[1])))
    return res

## $N=3$

In [ ]:
# def MMM(lam,nu1,nu2,nu3):
#     tmp  = epsilon(nu1)*epsilon(lam[1],-1)
#     tmp += epsilon(nu2)*epsilon(lam[2],-1)
#     tmp += epsilon(nu3)*epsilon(nu2,-1)
#     tmp += -u[0]/u[1]/q3*epsilon(lam[0])*epsilon(lam[1],-1)
#     tmp += -u[0]/u[2]/q3*epsilon(lam[0])*epsilon(lam[2],-1)
#     tmp += -u[1]/u[2]/q3*epsilon(lam[1])*epsilon(lam[2],-1)
#     tmp += u[1]/u[2]*epsilon(nu1)*epsilon(lam[2],-1)
#     tmp += u[0]/u[1]/q3*epsilon(lam[0])*epsilon(nu1,-1)
#     tmp += u[0]/u[2]/q3*epsilon(lam[0])*epsilon(nu2,-1)
#     tmp += u[1]/u[2]/q3*epsilon(lam[1])*epsilon(nu2,-1)
#     tmp += -u[1]/u[2]*epsilon(nu1)*epsilon(nu2,-1)
#     tmp += -u[1]/u[2]/q3*epsilon(nu1)*epsilon(nu2,-1)
#     tmp += u[1]/u[2]/q3*epsilon(nu1)*epsilon(nu3,-1)
#     res  = u[0]**(sum(lam[0])+sum(lam[1])+sum(lam[2])-sum(nu1)-sum(nu2))
#     res *= u[1]**(sum(nu1)+sum(nu2)-sum(nu3)) * u[2]**sum(nu3)
#     res *= (-q)**(sum(nu1)+sum(nu2)+sum(nu3))/DET(chi2d(nu1))/DET(chi2d(nu2))/DET(chi2d(nu3))
#     res *= b_tilde(nu1) * b_tilde(nu2) * b_tilde(nu3)
#     res *= PE((1-t)/(1-q)*tmp-3/(1-q1)/(1-q2))
#     return res

In [470]:
def MMM3(lam,nu1,nu2,nu3):
    res  = NekrasovJEB(nu1,lam[1],q3)
    res *= NekrasovJEB(nu2,lam[2],q3)
    res *= NekrasovJEB(nu3,nu2,q3)
    res /= NekrasovJEB(lam[0],lam[1],u[0]/u[1])
    res /= NekrasovJEB(lam[0],lam[2],u[0]/u[2])
    res /= NekrasovJEB(lam[1],lam[2],u[1]/u[2])
    res *= NekrasovJEB(nu1,lam[2],q3*u[1]/u[2])
    res *= NekrasovJEB(lam[0],nu1,u[0]/u[1])
    res *= NekrasovJEB(lam[0],nu2,u[0]/u[2])
    res *= NekrasovJEB(lam[1],nu2,u[1]/u[2])
    res /= NekrasovJEB(nu1,nu2,q3*u[1]/u[2])
    res /= NekrasovJEB(nu1,nu2,u[1]/u[2])
    res *= NekrasovJEB(nu1,nu3,u[1]/u[2])
    res *= u[0]**(sum(lam[0])+sum(lam[1])+sum(lam[2])-sum(nu1)-sum(nu2))
    res *= u[1]**(sum(nu1)+sum(nu2)-sum(nu3)) * u[2]**sum(nu3)
    res *= (-q)**(sum(nu1)+sum(nu2)+sum(nu3))/DET(chi2d(nu1))/DET(chi2d(nu2))/DET(chi2d(nu3))
    res *= b_tilde(nu1) * b_tilde(nu2) * b_tilde(nu3)
    return res

In [38]:
def MMM3(lam,nu1,nu2,nu3):
    res  = NekrasovJEB(nu1,lam[1],q3)
    res *= NekrasovJEB(nu2,lam[2],q3)
    res *= NekrasovJEB(nu3,nu2,q3)
    res /= NekrasovJEB(lam[0],lam[1],u[0]/u[1])
    res /= NekrasovJEB(lam[0],lam[2],u[0]/u[2])
    res /= NekrasovJEB(lam[1],lam[2],u[1]/u[2])
    res *= NekrasovJEB(nu1,lam[2],q3*u[1]/u[2])
    res *= NekrasovJEB(lam[0],nu1,u[0]/u[1])
    res *= NekrasovJEB(lam[0],nu2,u[0]/u[2])
    res *= NekrasovJEB(lam[1],nu2,u[1]/u[2])
    res /= NekrasovJEB(nu1,nu2,q3*u[1]/u[2])
    res /= NekrasovJEB(nu1,nu2,u[1]/u[2])
    res *= NekrasovJEB(nu1,nu3,u[1]/u[2])
    res /= NekrasovJEB(nu1,nu1,1)
    res /= NekrasovJEB(nu2,nu2,1)
    res /= NekrasovJEB(nu3,nu3,1)
    res *= u[0]**(sum(lam[0])+sum(lam[1])+sum(lam[2])-sum(nu1)-sum(nu2))
    res *= u[1]**(sum(nu1)+sum(nu2)-sum(nu3))
    res *= u[2]**sum(nu3)
    res *= q3**-(sum(nu1)+sum(nu2)+sum(nu3))
    return res

In [39]:
def ker3(lam,nu1,nu2,nu3):
    X1 = tensor([p[1],p[0],p[0]])
    X2 = tensor([p[0],p[1],p[0]])
    X3 = tensor([p[0],p[0],p[1]])
    tmp  = (u[0]*epsilon(lam[0])+u[1]*epsilon(lam[1])+u[2]*epsilon(lam[2])-q3*u[1]*epsilon(nu1)-q3*u[2]*epsilon(nu2))*X1
    tmp += (u[1]*epsilon(nu1)+u[2]*epsilon(nu2)-q3*u[2]*epsilon(nu3))*X2
    tmp += u[2]*epsilon(nu3)*X3
    return tmp

In [40]:
def Whittaker3(lam,n):
    if n == 0:
        return sum(sum(sum(MMM3(lam,nu1,nu2,nu3) for nu3 in subpart(nu2)) for nu2 in subpart(lam[2])) for nu1 in subpart(lam[1])) * tensor([p[0]]*3)
    return sum(sum(sum(MMM3(lam,nu1,nu2,nu3)*s[n]((1-t)/(1-q)*ker3(lam,nu1,nu2,nu3)) for nu3 in subpart(nu2)) for nu2 in subpart(lam[2])) for nu1 in subpart(lam[1]))

## $N$

In [130]:
def framing_on_LSF(x,power=1):
    N = len(x[0].parent().tensor_factors())
    LSF = LazySymmetricFunctions(tensor([p]*N))
    return LSF(lambda n: framing_on_tensor(x[n],power), valuation=0)

In [131]:
def VV(x):
    N = len(x.parent().tensor_factors())
    X = [tensor([p[1] if j==i else p.one() for j in range(N)]) for i in range(N)]
    x = framing_on_tensor(x)
    x = diag_plethysm(x,[p[1]/t-r*q3^i*epsilon([]) for i in range(N)])
    x = exp( LSF(lambda n: (-1)^n*p[n](sum(X)/(1-q))/n, valuation=1) ) * x
    return framing_on_LSF(x.map_coefficients(lambda c: c.subs(r=-1)))

In [173]:
def MMM(lam,nu,shift=0):
    N = len(lam)
    vu = [u[a+shift] for a in range(N)]
    res = vu[0]**sum(map(sum,lam))
    res *= prod((vu[0])**-sum(k) for k in nu)
    res *= prod(NekrasovJEB(nu[b],lam[a],q3*vu[b+1]/vu[a])/NekrasovJEB(lam[b],lam[a],vu[b]/vu[a]) for a in range(N) for b in range(a))
    res *= prod(NekrasovJEB(lam[b],nu[a],vu[b]/vu[a+1])/NekrasovJEB(nu[b],nu[a],q3*vu[b+1]/vu[a+1]) for a in range(N-1) for b in range(a+1))
    return res

In [178]:
def Whittaker(lam,n,shift=0):
    N = len(lam)
    vu = [u[a+shift] for a in range(N)]
    one = tensor([p.one()]*(N+shift))
    zero = tensor([p.zero()]*(N+shift))
    X = [tensor([p.one()]*shift+[p[1] if j==i else p.one() for j in range(N)]) for i in range(N)]
    if N==0:
        return one if n==0 else zero
    else:
        nulist = list(list(subpart(lam[a])) for a in range(1,N))
        return sum( MMM(lam,nu,shift) * sum(evalArg(s[k],(1-t)/(1-q)*(sum(vu[a]*epsilon(lam[a]) for a in range(N))-sum(q3*vu[a]*epsilon(nu[a-1]) for a in range(1,N)))*X[0]) * Whittaker(nu,n-k,shift+1) for k in range(n+1)) for nu in product(*nulist))

In [179]:
def testGHT(lam):
    N = len(lam)
    X = [tensor([p[1] if j==i else p[[]] for j in range(N)]) for i in range(N)]
    LSF = LazySymmetricFunctions(tensor([p]*N))
    rhs = exp(LSF( lambda n: (-1)^n*p[n]((1-t)/(1-q)*epsilon([])*sum(X))/n , valuation=1)) * LSF(lambda n: framing_on_tensor(Whittaker(lam,n),power=-1), valuation=0)
    return (GMK(lam) - rhs).truncate(sum(map(sum,lam))) == 0

In [ ]:
all(testGHT(lam) for d in range(4) for N in range(2,7) for lam in mPartitions(N,d))